In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.database.queries import scenario_stats
from src.genai.prompt_templates import SYSTEM_PROMPT, build_whatif_prompt
from src.genai.llm_client import ask_llm

print(ask_llm("Say hello in one short sentence."))

Hello!


In [2]:
from src.database.queries import mode_performance, region_mode_risk
from src.genai.prompt_templates import SYSTEM_PROMPT, build_briefing_prompt
prompt=build_briefing_prompt(mode_performance(), region_mode_risk())
print(prompt)
print("-"*60)
print(ask_llm(prompt, system_prompt=SYSTEM_PROMPT,max_tokens=600))

Write a short risk briefing from this data.

Shipping mode performance:
 shipping_mode  total_orders  late_pct  avg_real_days  avg_scheduled_days  reliability_rank
Standard Class        103151      39.8           3.99                 4.0                 1
      Same Day          9293      47.9           0.48                 0.0                 2
  Second Class         33806      79.8           3.99                 2.0                 3
   First Class         26512     100.0           2.00                 1.0                 4

Highest-risk region and shipping mode combinations:
   order_region shipping_mode  total_orders  late_pct
      Caribbean   First Class          1132     100.0
Central America   First Class          4185     100.0
    East of USA   First Class          1050     100.0
   Eastern Asia   First Class           989     100.0
 Eastern Europe   First Class           632     100.0

Write the briefing now.
------------------------------------------------------------
**Ris

In [3]:
from src.genai.llm_client import get_client

for m in get_client().models.list().data:
    print(m.id)

meta-llama/llama-prompt-guard-2-86m
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
whisper-large-v3
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
openai/gpt-oss-20b


In [4]:
for v in ["v1", "v2"]:
    try:
        p = build_briefing_prompt(mode_performance(), region_mode_risk(), version=v)
    except KeyError:
        print(f"Skipping {v}: no prompt template is defined.")
        continue
    print(f"===== {v} =====")
    print(ask_llm(p, system_prompt=SYSTEM_PROMPT, max_tokens=1500))
    print()

===== v1 =====
**Risk Briefing – Shipping Performance Overview**

**1. Shipping‑mode reliability (overall)**  
| Shipping Mode | Total Orders | Late % | Avg Real Days | Avg Scheduled Days | Reliability Rank |
|---------------|--------------|--------|---------------|--------------------|------------------|
| Standard Class | 103,151 | 39.8 % | 3.99 | 4.0 | 1 (best) |
| Same Day | 9,293 | 47.9 % | 0.48 | 0.0 | 2 |
| Second Class | 33,806 | 79.8 % | 3.99 | 2.0 | 3 |
| First Class | 26,512 | **100 %** | 2.00 | 1.0 | 4 (worst) |

*Key take‑aways*  
- **Standard Class** delivers the highest volume with the lowest lateness (39.8 %).  
- **First Class** shows a perfect lateness rate (100 %) despite the shortest scheduled window (1 day).  
- **Same Day** and **Second Class** have moderate‑to‑high lateness (47.9 % and 79.8 % respectively).  

**2. Highest‑risk region & mode combinations (First Class)**  
All listed regions use First Class and exhibit a 100 % late rate.

| Region | Shipping Mode 

In [5]:
import importlib
import src.genai.prompt_templates as pt
importlib.reload(pt)
print(pt.__file__)
print([n for n in dir(pt) if n.isupper() or n.startswith("build")])

c:\Users\Rishj\OneDrive\Desktop\PROJECTS\supply_chain_intelligence\src\genai\prompt_templates.py
['BRIEFING_V1', 'BRIEFING_V2', 'BRIEFING_V3', 'SYSTEM_PROMPT', 'WHATIF_V1', 'WHATIF_V2', 'WHATIF_V3', 'build_briefing_prompt', 'build_facts', 'build_whatif_facts', 'build_whatif_prompt']


In [6]:
from src.database.queries import mode_performance, region_mode_risk
from src.genai.prompt_templates import SYSTEM_PROMPT, build_briefing_prompt
from src.genai.llm_client import ask_llm

p = build_briefing_prompt(mode_performance(), region_mode_risk(), version="v2")
print(p)
print("=" * 60)
print(ask_llm(p, system_prompt=SYSTEM_PROMPT, max_tokens=1500))

Write a risk briefing for an operations manager.

DATA
Shipping mode performance:
 shipping_mode  total_orders  late_pct  avg_real_days  avg_scheduled_days  reliability_rank
Standard Class        103151      39.8           3.99                 4.0                 1
      Same Day          9293      47.9           0.48                 0.0                 2
  Second Class         33806      79.8           3.99                 2.0                 3
   First Class         26512     100.0           2.00                 1.0                 4

PRE-CALCULATED FACTS (use these exactly; do not calculate anything yourself):
- Worst mode: First Class (100.0% late; promised 1.0 days, actual 2.0 days, gap 1.00 days)
- Best mode: Standard Class (39.8% late)
- Overall late rate across 172,762 order items: 57.3%
- 16 region/mode combinations are 100% late, so the problem is not specific to one region

RULES
- Use only numbers that appear above. Do no arithmetic.
- 'Late' means delivered after the sched

In [7]:
from src.genai.llm_client import get_client
from src.genai.prompt_templates import SYSTEM_PROMPT

client = get_client()
r = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": p},
    ],
    temperature=0.2,
    max_tokens=1500,
)
print("finish_reason:", r.choices[0].finish_reason)
print("content:", repr(r.choices[0].message.content))
print("usage:", r.usage)

finish_reason: length
content: '**Headline:** First Class Shipping Mode Exhibits 100% Late Rate, Raising Operational Risk.  \nLate deliveries are defined as shipments arriving after the scheduled date.  \n\n**Key risks**  \n- First Class mode has 100.0% late rate, with a 1.00 day delivery gap.  \n- Overall late rate is 57.3% across 172,762 order items.  \n- '
usage: CompletionUsage(completion_tokens=1500, prompt_tokens=412, total_tokens=1912, completion_time=1.6577130310000001, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=1409), prompt_time=0.025166277, prompt_tokens_details=None, queue_time=0.20975538, total_time=1.682879308)


In [8]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import src.genai.prompt_templates as pt
print("BRIEFING_V3" in dir(pt))
print(Path(pt.__file__).read_text().count("def build_briefing_prompt"))

True
3


In [9]:
from src.database.queries import mode_performance, region_mode_risk
from src.genai.prompt_templates import SYSTEM_PROMPT, build_briefing_prompt
from src.genai.llm_client import ask_llm

In [10]:
import itertools
mode_df, region_df = mode_performance(), region_mode_risk()
for model, v in itertools.product(
    ["openai/gpt-oss-20b", "openai/gpt-oss-120b"], ["v2", "v3"]
):
    p = build_briefing_prompt(mode_df, region_df, version=v)
    out = ask_llm(p, system_prompt=SYSTEM_PROMPT, model=model, max_tokens=2000)
    print(f"##### {model} | {v} | {len(out.split())} words #####")
    print(out)
    print()

##### openai/gpt-oss-20b | v2 | 76 words #####
**Headline:**  
High Late Delivery Rates Across Shipping Modes Demand Immediate Action.

**Key risks:**  
- First Class mode shows 100.0% late deliveries, doubling the promised 1.0 days to 2.0 days.  
- Overall late rate is 57.3% across 172,762 order items, indicating systemic delays.  
- Sixteen region/mode combinations are 100.0% late, showing widespread issues.

**Recommended actions:**  
- Investigate First Class shipping to reduce the 1.0 day delay.  
- Reassess Standard Class processes to maintain the 39.8% late rate.

##### openai/gpt-oss-20b | v3 | 72 words #####
Headline: First Class misses its promise on every order, driving overall lateness to 57.3%.

Key risks  
- First Class: 100.0 % late; promised 1.0 days, actual 2.0 days.  
- Second Class: 79.8 % late; promised 2.0 days, actual 3.99 days.  
- Overall late rate is 57.3 % across 172,762 order items.

Recommended actions  
- Extend First Class promised delivery window to 2.0 d

In [11]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.database.queries import scenario_stats
from src.genai.prompt_templates import SYSTEM_PROMPT, build_whatif_prompt
from src.genai.llm_client import ask_llm
region, from_mode, to_mode = "Central America", "First Class", "Standard Class"
question = f"What if we move all {from_mode} orders in {region} to {to_mode}?"

a, b = scenario_stats(region, from_mode, to_mode)

for v in ["v1", "v2"]:
    p = build_whatif_prompt(question, region, from_mode, to_mode, a, b, version=v)
    print(f"===== {v} =====")
    if v == "v2":
        print(p)
        print("-" * 40)
    print(ask_llm(p, system_prompt=SYSTEM_PROMPT, max_tokens=2000))
    print()

===== v1 =====
**Impact of moving all 4,185 First‑Class orders in Central America to Standard‑Class**

| Metric | Current (First‑Class) | After move (Standard‑Class) | Net change |
|--------|----------------------|-----------------------------|------------|
| **Late‑order items** | 4,185 (100 % of the orders) | 1,678 (40.1 % of the orders) | **‑2,507 late items** |
| **Late‑order rate** | 100 % | 40.1 % | **‑59.9 percentage points** |
| **Promised delivery time** | 1.0 day | 4.0 days | **+3.0 days** (longer promise) |
| **Actual average delivery time** | 2.0 days | 4.0 days | **+2.0 days** |
| **Average delivery‑time increase** | – | – | **+2.00 days** (from 2.0 to 4.0 days) |

### What this means for the operation
- **Late‑order reduction:** You would eliminate 2,507 late deliveries, cutting the late‑order count by more than half.
- **Service‑level shift:** The on‑time performance would improve dramatically (from 0 % on‑time to about 60 % on‑time), but the promised delivery window wou

In [12]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.database.queries import scenario_stats
from src.genai.prompt_templates import SYSTEM_PROMPT, build_whatif_prompt
from src.genai.llm_client import ask_llm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
scenarios = [
    ("Central America", "First Class", "Standard Class", 1.0),
    ("Western Europe", "Second Class", "Standard Class", 0.5),
]

for region, f, t, share in scenarios:
    q = f"What if we move {share:.0%} of {f} orders in {region} to {t}?"
    a, b = scenario_stats(region, f, t)
    for v in ["v2", "v3"]:
        p = build_whatif_prompt(q, region, f, t, a, b, share=share, version=v)
        print(f"##### {region}: {f} -> {t} ({share:.0%}) | {v} #####")
        print(ask_llm(p, system_prompt=SYSTEM_PROMPT, max_tokens=2000))
        print()

##### Central America: First Class -> Standard Class (100%) | v2 #####
Answer: Mixed.  
Impact:  
- Late order items decrease by 2,507 (from 4,185 to 1,678).  
- Average delivery time increases by 2.00 days (from 2.0 to 4.0).  
Caveat: The result assumes the moved orders would perform like Standard Class’s historical average in Central America.

##### Central America: First Class -> Standard Class (100%) | v3 #####
Answer: Shifting 100% of Central America First Class orders to Standard Class trades the current 100% lateness on a 1‑day promise for a reduced 40.1% lateness on a 4‑day promise, adding two days to delivery time.  
Impact:  
- Late order items: 4,185 → 1,678  
- Average delivery time (actual): 2.0 days → 4.0 days (+2.00 days)  
Caveat: The result assumes the moved orders would perform like Standard Class’s historical average in this region.

##### Western Europe: Second Class -> Standard Class (50%) | v2 #####
Answer: Better.  
Impact:  
- Late order items decrease by 1,072 